In [ ]:
#Import the necessary libraries
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from scipy.stats import qmc
import matplotlib.pyplot as plt

In [ ]:
X = np.array([[0.66579958, 0.12396913],
              [0.87779099, 0.7786275 ],
              [0.14269907, 0.34900513],
              [0.84527543, 0.71112027],
              [0.45464714, 0.29045518],
              [0.57771284, 0.77197318],
              [0.43816606, 0.68501826],
              [0.34174959, 0.02869772],
              [0.33864816, 0.21386725],
              [0.70263656, 0.9265642 ],
              [0.073440, 0.995966],
              [0.286637, 0.863817],
              [0.76404438, 0.93734368],
              [0.697156, 0.046277],
              [0.713534, 0.622983],
              [0.860214, 0.472626],
              [0.682311, 0.700075],
              [0.865398, 0.167491],
              [0.876322, 0.451875],
              [0.753330, 0.030819],
              [0.840484, 0.366788],
              [0.693369, 0.316304]

])
y = np.array([ 0.53899612, 0.42058624, -0.06562362, 0.29399291,
               0.21496451, 0.02310555, 0.24461934, 0.03874902,
               -0.01385762, 0.61120522, 0.545876278176769, 0.064657824682165,
               0.175664369696048, 0.661273278850939, 0.5281368705482168,
               0.7843694604014655, 0.571599983683343, 0.4183161117651154,
               0.6773369360993695, 0.294629069692715, 0.3762580777463512,
               0.5698036470501392

])

#shape of X & y
print(X.shape)
print(y.shape)

In [ ]:
#GP setup
kernel = ConstantKernel(1.0) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0],
    length_scale_bounds=(1e-3, 1e4)
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=20,
    alpha=1e-6,
    normalize_y=True
)

# Fit on your 2D data
gpr.fit(X, y.reshape(-1, 1)) # Reshape y1 to (N, 1) for fitting

# Check what the GP learned
print("GP Model Diagnostics:")
print(f"  Kernel: {gpr.kernel_}")
print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
print(f"  Training score: {gpr.score(X, y.reshape(-1, 1)):.3f}") # Reshape y1 for scoring

# Define bounds for 2D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max())   # x2 bounds
]
print(f"  Bounds: {bounds}")

# Generate 2D candidates using Latin Hypercube
sampler = qmc.LatinHypercube(d=2)
X_candidates = qmc.scale(
    sampler.random(n=5000),
    l_bounds=[bounds[0][0], bounds[1][0]],
    u_bounds=[bounds[0][1], bounds[1][1]]
)

# Predict on candidates
y_pred, y_std = gpr.predict(X_candidates, return_std=True)

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]

print(f"\nNext Point to Sample:")
print(f"  X = {x_next}")
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}, "
          f"pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")
    # add X candidate to X
    X = np.vstack((X, X_candidates[idx]))
    # Reshape y to (N, 1) if it's not already, and reshape the new prediction to (1, 1)
    # It's better to ensure y1 is always (N, 1) from the start, but for an immediate fix:
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    y = np.vstack((y, y_pred[idx].reshape(-1, 1)))

Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 2D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max())   # x2 bounds
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )


y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")